[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/03-feature-detection.ipynb)

# Module 6.3 — Feature Detection
**Module 6: Computer Vision** | Estimated time: 25 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Detect edges using the Canny algorithm and understand its threshold parameters
- Compute image gradients with Sobel and Laplacian operators
- Detect corners with Harris corner detector and `cv2.goodFeaturesToTrack`
- Extract keypoints and descriptors using SIFT and ORB
- Match features between two images using BFMatcher with the ratio test
- Visualise matches with `cv2.drawMatches`

In [ ]:
!pip install opencv-python-headless --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests, os

print(f'OpenCV {cv2.__version__}')

os.makedirs('/tmp/cv_features', exist_ok=True)

def show(img, title='', bgr=True, figsize=(7, 5)):
    plt.figure(figsize=figsize)
    if bgr and len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cmap = 'gray' if len(img.shape) == 2 else None
    plt.imshow(img, cmap=cmap)
    plt.title(title); plt.axis('off')
    plt.tight_layout(); plt.show()

def show_pair(img1, img2, t1='', t2='', figsize=(14, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, im, t in zip(axes, [img1, img2], [t1, t2]):
        cmap = 'gray' if len(im.shape) == 2 else None
        if len(im.shape) == 3:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        ax.imshow(im, cmap=cmap); ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()

# Download two versions of the same scene for matching
urls = {
    'scene1.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikesgray.jpg/320px-Bikesgray.jpg',
    'scene2.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikesgray.jpg/280px-Bikesgray.jpg',
}
for fname, url in urls.items():
    r = requests.get(url)
    path = f'/tmp/cv_features/{fname}'
    with open(path, 'wb') as f:
        f.write(r.content)
print('Images downloaded.')

img1_bgr = cv2.imread('/tmp/cv_features/scene1.jpg')
img2_bgr = cv2.imread('/tmp/cv_features/scene2.jpg')
if img1_bgr is None:
    img1_bgr = np.random.randint(50, 200, (300, 400, 3), dtype=np.uint8)
    cv2.rectangle(img1_bgr, (50, 50), (150, 150), (200, 100, 50), 3)
    cv2.circle(img1_bgr, (250, 150), 60, (50, 100, 200), 3)
    img2_bgr = cv2.resize(img1_bgr, (320, 240))
img1_gray = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
img2_gray = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2GRAY)
print(f'Image 1: {img1_bgr.shape}   Image 2: {img2_bgr.shape}')

## Edge Detection: Canny Algorithm

The Canny edge detector is the gold standard for edge detection. It works in four stages:
1. **Gaussian blur** — suppress noise
2. **Gradient computation** — find intensity changes
3. **Non-maximum suppression** — thin edges to 1-pixel width
4. **Hysteresis thresholding** — keep strong edges and edges connected to them

The two thresholds control which gradient magnitudes are considered edges:
- `threshold1` (low): edges below this are discarded
- `threshold2` (high): edges above this are always kept

In [ ]:
gray = img1_gray.copy()

# Canny with different threshold pairs
edges_tight  = cv2.Canny(gray,  50, 150)   # fewer edges
edges_medium = cv2.Canny(gray, 100, 200)
edges_loose  = cv2.Canny(gray,  30,  80)   # more edges (more noise too)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, im, t in zip(axes,
                     [gray, edges_tight, edges_medium, edges_loose],
                     ['Grayscale', 'Canny (50,150)', 'Canny (100,200)', 'Canny (30,80)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Canny Edge Detection — Threshold Tuning', fontweight='bold')
plt.tight_layout(); plt.show()

# Auto-threshold using the median pixel value
v = np.median(gray)
low  = int(max(0,   0.66 * v))
high = int(min(255, 1.33 * v))
edges_auto = cv2.Canny(gray, low, high)
print(f'Auto thresholds: low={low}, high={high}')
show(edges_auto, f'Canny — Auto ({low}, {high})')

## Gradient Operators: Sobel and Laplacian

**Sobel** computes the gradient in X or Y direction separately, then we combine them. It is directional — you can detect horizontal or vertical edges selectively.

**Laplacian** computes the second derivative, detecting edges in all directions simultaneously. It is more sensitive to noise.

In [ ]:
blurred = cv2.GaussianBlur(gray, (3, 3), 0)

# Sobel in X direction (detects vertical edges)
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_x_abs = cv2.convertScaleAbs(sobel_x)

# Sobel in Y direction (detects horizontal edges)
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel_y_abs = cv2.convertScaleAbs(sobel_y)

# Combined magnitude
sobel_combined = cv2.addWeighted(sobel_x_abs, 0.5, sobel_y_abs, 0.5, 0)

# Laplacian
laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
laplacian_abs = cv2.convertScaleAbs(laplacian)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, im, t in zip(axes,
    [gray, sobel_x_abs, sobel_y_abs, sobel_combined, laplacian_abs],
    ['Original', 'Sobel X', 'Sobel Y', 'Sobel Combined', 'Laplacian']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Gradient Operators', fontweight='bold')
plt.tight_layout(); plt.show()

## Corner Detection

Corners are interest points where intensity changes significantly in multiple directions. They are stable features for tracking and matching.

**Harris Corner Detector** — measures corner strength via eigenvalues of the gradient matrix. Outputs a response map: high values = likely corners.

**Shi-Tomasi** (`cv2.goodFeaturesToTrack`) — an improvement on Harris that picks the top-N strongest corners directly. Widely used for optical flow.

In [ ]:
gray_f = np.float32(gray)

# Harris corner response
harris = cv2.cornerHarris(gray_f, blockSize=2, ksize=3, k=0.04)
harris_dilated = cv2.dilate(harris, None)   # enhance corner visibility

# Threshold and mark corners
harris_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
harris_img[harris_dilated > 0.01 * harris_dilated.max()] = [0, 0, 255]

# Shi-Tomasi: pick top 200 corners
corners = cv2.goodFeaturesToTrack(gray, maxCorners=200,
                                  qualityLevel=0.01, minDistance=10)
shi_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
if corners is not None:
    for pt in corners.reshape(-1, 2):
        x, y = int(pt[0]), int(pt[1])
        cv2.circle(shi_img, (x, y), 4, (0, 255, 0), -1)
print(f'Shi-Tomasi corners found: {len(corners) if corners is not None else 0}')

show_pair(harris_img, shi_img,
          'Harris Corners (red)', f'Shi-Tomasi ({len(corners) if corners is not None else 0} corners)')

## SIFT and ORB Features

**SIFT** (Scale-Invariant Feature Transform) detects and describes keypoints that are invariant to scale and rotation. It is accurate but was historically patent-protected (now open since 2020 and available in OpenCV 4.4+).

**ORB** (Oriented FAST and Rotated BRIEF) is a fast, patent-free alternative that combines the FAST keypoint detector with the BRIEF descriptor. It is much faster than SIFT and good enough for real-time applications.

In [ ]:
# --- SIFT ---
try:
    sift = cv2.SIFT_create(nfeatures=500)
    kp_sift, des_sift = sift.detectAndCompute(gray, None)
    sift_img = cv2.drawKeypoints(
        gray, kp_sift, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    print(f'SIFT keypoints: {len(kp_sift)}')
except cv2.error as e:
    print(f'SIFT unavailable: {e}')
    sift_img = gray.copy()
    kp_sift, des_sift = [], None

# --- ORB ---
orb = cv2.ORB_create(nfeatures=500)
kp_orb, des_orb = orb.detectAndCompute(gray, None)
orb_img = cv2.drawKeypoints(
    gray, kp_orb, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
print(f'ORB keypoints : {len(kp_orb)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(sift_img, cmap='gray' if len(sift_img.shape) == 2 else None)
axes[0].set_title(f'SIFT Keypoints ({len(kp_sift)})')
axes[0].axis('off')
axes[1].imshow(orb_img, cmap='gray' if len(orb_img.shape) == 2 else None)
axes[1].set_title(f'ORB Keypoints ({len(kp_orb)})')
axes[1].axis('off')
plt.suptitle('SIFT vs ORB Feature Detection', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nKey difference:')
print('  SIFT → 128-dim float descriptor, scale+rotation invariant, slower')
print('  ORB  → 256-bit binary descriptor, very fast, good for real-time')

## Feature Matching with BFMatcher

**BFMatcher** (Brute-Force Matcher) compares every descriptor in image 1 against every descriptor in image 2 and returns the best matches.

The **Lowe ratio test** filters out ambiguous matches: a match is kept only if the distance to the best match is significantly smaller than the distance to the second-best match (typically ratio < 0.75).

In [ ]:
# Detect ORB features in both images
orb2 = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb2.detectAndCompute(img1_gray, None)
kp2, des2 = orb2.detectAndCompute(img2_gray, None)

if des1 is None or des2 is None:
    print('Not enough features detected — using synthetic data')
else:
    # BFMatcher with Hamming distance (for binary descriptors like ORB)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

    # knnMatch returns k=2 nearest neighbours
    knn_matches = bf.knnMatch(des1, des2, k=2)

    # Lowe ratio test (keep only unambiguous matches)
    good_matches = []
    for m, n in knn_matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    print(f'Total raw matches : {len(knn_matches)}')
    print(f'After ratio test  : {len(good_matches)}')
    print(f'Match quality     : {len(good_matches)/len(knn_matches)*100:.1f}%')

    # Draw matches
    match_img = cv2.drawMatches(
        img1_gray, kp1,
        img2_gray, kp2,
        good_matches[:50], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    plt.figure(figsize=(16, 6))
    plt.imshow(match_img, cmap='gray')
    plt.title(f'Feature Matches: {len(good_matches)} good matches (showing top 50)')
    plt.axis('off')
    plt.tight_layout(); plt.show()

## Summary

| Technique | Function | Use case |
|---|---|---|
| Canny edges | `cv2.Canny(img, t1, t2)` | General edge detection |
| Sobel gradient | `cv2.Sobel(img, cv2.CV_64F, dx, dy)` | Directional edges |
| Harris corners | `cv2.cornerHarris()` | Corner strength map |
| Shi-Tomasi | `cv2.goodFeaturesToTrack()` | Top-N corners (tracking) |
| SIFT features | `cv2.SIFT_create()` | Accurate, scale-invariant |
| ORB features | `cv2.ORB_create()` | Fast, real-time matching |
| BFMatcher | `cv2.BFMatcher()` + ratio test | Feature matching |

## Practice Exercises

**Exercise 1 — Interactive Canny:**  
Write a function that accepts `low_threshold` and `high_threshold` as parameters and plots Canny edges for values in `[(50,150), (80,160), (100,200), (150,250)]`. Which settings work best for your image?

**Exercise 2 — Keypoint Comparison Table:**  
Run both SIFT and ORB on the same image and record: number of keypoints, average keypoint size, and the time taken using `time.time()`. Display results in a formatted table using `pandas`.

**Exercise 3 — Homography from Matches:**  
Use the good matches between two images to compute a homography matrix with `cv2.findHomography(pts1, pts2, cv2.RANSAC)`. Use `cv2.warpPerspective()` to align image 2 to image 1's perspective. Display the warped result.